# Proyecto de Business Intelligence — COVID-19 Global Dataset
## Fase 4: Definición de KPIs

**Dataset fuente:** [Corona Virus Report — Kaggle (imdevskp)](https://www.kaggle.com/datasets/imdevskp/corona-virus-report)  
**Archivo utilizado:** `covid_19_clean_complete.csv`  

---

### Contexto de continuidad

Esta fase utiliza directamente el modelo de datos construido en la Fase 3:

| Tabla | Descripción |
|---|---|
| `hechos_covid_diario.csv` | Tabla de hechos: granularidad País/Provincia × Día |
| `dim_pais_snapshot.csv` | Snapshot por país al cierre con métricas derivadas |
| `dim_region.csv` | Agregados por Región OMS |
| `dim_fecha.csv` | Dimensión calendario |

Los 7 KPIs definidos aquí se enlazan directamente con las **hipótesis validadas en la Fase 2** (H1: CFR difiere entre regiones, confirmada; H2: timing correlaciona con CFR, no confirmada) y con las **3 preguntas clave de la Fase 1**.

---
## 1. Configuración e Insumos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi'       : 120,
    'font.family'      : 'DejaVu Sans',
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
    'axes.labelsize'   : 11,
    'xtick.labelsize'  : 9,
    'ytick.labelsize'  : 9,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})

ALPHA  = 0.05
COLORS = ['steelblue', 'crimson', 'seagreen', 'darkorange']

# ── Reconstrucción del pipeline de Fase 3 ────────────────────────────────────
# (garantiza que este notebook sea autocontenido y reproducible)
df_raw = pd.read_csv('../Data/covid_19_clean_complete.csv', parse_dates=['Date'])

latest_date = df_raw['Date'].max()
latest      = df_raw[df_raw['Date'] == latest_date]

# Snapshot por país
ct = (
    latest.groupby('Country/Region')[['Confirmed','Deaths','Recovered','Active']]
    .sum().reset_index()
)
ct = ct[ct['Confirmed'] >= 500].copy()
ct['CFR'] = ct['Deaths'] / ct['Confirmed'] * 100
ct_r = ct.merge(
    df_raw[['Country/Region','WHO Region']].drop_duplicates(), on='Country/Region'
)

# Timing
first_case = (
    df_raw[df_raw['Confirmed'] > 0]
    .groupby('Country/Region')['Date'].min()
    .reset_index().rename(columns={'Date':'First_Case_Date'})
)
first_case['Days_from_start'] = (
    first_case['First_Case_Date'] - first_case['First_Case_Date'].min()
).dt.days

# df_pais: snapshot enriquecido
df_pais = ct_r.copy()
df_pais['Recovery_Rate'] = (df_pais['Recovered'] / df_pais['Confirmed'] * 100).round(4)
df_pais['Active_Rate']   = (df_pais['Active']    / df_pais['Confirmed'] * 100).round(4)
df_pais['IMR']           = np.where(
    df_pais['Recovered'] > 0,
    df_pais['Deaths'] / df_pais['Recovered'] * 100, np.nan
).round(4)
df_pais = df_pais.merge(
    first_case[['Country/Region','First_Case_Date','Days_from_start']],
    on='Country/Region', how='left'
)
df_pais['Epidemic_Size'] = pd.qcut(
    df_pais['Confirmed'], q=4,
    labels=['Bajo (Q1)','Moderado (Q2)','Alto (Q3)','Crítico (Q4)'],
    duplicates='drop'
)

# df_region
df_region = df_pais.groupby('WHO Region').agg(
    Paises      =('Country/Region','count'),
    Confirmados =('Confirmed','sum'),
    Muertes     =('Deaths','sum'),
    Recuperados =('Recovered','sum'),
    Activos     =('Active','sum'),
).reset_index()
df_region['CFR_region']      = (df_region['Muertes']     / df_region['Confirmados'] * 100).round(4)
df_region['Recovery_region'] = (df_region['Recuperados'] / df_region['Confirmados'] * 100).round(4)
df_region['Active_region']   = (df_region['Activos']     / df_region['Confirmados'] * 100).round(4)

# df_hechos (con nuevos diarios y MA7)
df_hechos = df_raw.copy()
df_hechos['Province/State'] = df_hechos['Province/State'].fillna(df_hechos['Country/Region'])
df_hechos['WHO Region']     = df_hechos['WHO Region'].str.strip()
for col in ['Confirmed','Deaths','Recovered','Active']:
    df_hechos[col] = df_hechos[col].clip(lower=0)
df_hechos = df_hechos.sort_values(['Country/Region','Province/State','Date'])
for col in ['Confirmed','Deaths','Recovered']:
    df_hechos[f'New_{col}'] = (
        df_hechos.groupby(['Country/Region','Province/State'])[col]
        .diff().clip(lower=0)
    )
df_hechos['MA7_New_Confirmed'] = (
    df_hechos.groupby(['Country/Region','Province/State'])['New_Confirmed']
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
).round(2)
df_hechos['MA7_New_Deaths'] = (
    df_hechos.groupby(['Country/Region','Province/State'])['New_Deaths']
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
).round(2)

# Totales globales
TOTAL_CONFIRMADOS = df_pais['Confirmed'].sum()
TOTAL_MUERTES     = df_pais['Deaths'].sum()
TOTAL_RECUPERADOS = df_pais['Recovered'].sum()

print(f' Insumos cargados.')
print(f'   Países en análisis : {len(df_pais)}')
print(f'   Fecha de corte     : {latest_date.date()}')
print(f'   Total confirmados  : {TOTAL_CONFIRMADOS:,}')
print(f'   Total muertes      : {TOTAL_MUERTES:,}')
print(f'   Total recuperados  : {TOTAL_RECUPERADOS:,}')

---
## 2. Marco de KPIs

Antes de definir cada indicador, se establece el marco que los conecta con el problema de negocio, las hipótesis y las preguntas clave del proyecto.

| KPI | Dimensión de análisis | Hipótesis | Pregunta clave |
|---|---|---|---|
| KPI 1 — CFR Global y por Región | Efectividad del sistema de salud | H1 (confirmada) | P1: disparidad en letalidad |
| KPI 2 — Tasa de Recuperación | Capacidad de atención médica | H1 (confirmada) | P3: gestión de recuperación |
| KPI 3 — Tasa de Casos Activos | Carga sanitaria simultánea | H1 (confirmada) | P1: presión sobre el sistema |
| KPI 4 — Índice de Mortalidad Relativa | Eficiencia clínica comparada | H1, H2 | P1, P3 |
| KPI 5 — Velocidad de la primera ola | Dinámica temporal de propagación | H2 (no confirmada) | P2: oleadas y aceleración |
| KPI 6 — Media móvil de nuevos casos | Tendencia de corto plazo | H2 | P2: períodos de mayor aceleración |
| KPI 7 — Concentración de la carga global | Equidad en la distribución pandémica | H1, H2 | P1, P2 |

---
## KPI 1 — Tasa de Fatalidad de Casos (CFR)

**Definición:** Porcentaje de casos confirmados que resultaron en fallecimiento al corte del dataset.  
**Fórmula:**  
$$\text{CFR} = \frac{\text{Deaths}}{\text{Confirmed}} \times 100$$

**Justificación de negocio:**  
El CFR es el indicador epidemiológico central de cualquier análisis de pandemia. Permite evaluar la letalidad relativa entre países y regiones, orientar la priorización de recursos hospitalarios y de UCI, y comparar la efectividad de los sistemas de salud. Un CFR elevado puede indicar subdiagnóstico (pocos tests), saturación hospitalaria o ausencia de tratamientos oportunos.

**Relación con hipótesis:**  
Es la variable de respuesta directa de **H1** (ANOVA confirmado, p = 0.046): la CFR *sí* difiere significativamente entre regiones de la OMS. También es la variable dependiente de **H2** (correlación con timing, p = 0.055).

**Umbral de referencia:** CFR < 2% = respuesta efectiva; 2–5% = alerta; > 5% = crítico.

In [ ]:
# ── Cómputo del KPI 1 ────────────────────────────────────────────────────────
cfr_global = TOTAL_MUERTES / TOTAL_CONFIRMADOS * 100

kpi1_region = df_region[['WHO Region','Confirmados','Muertes','CFR_region']].copy()
kpi1_region = kpi1_region.sort_values('CFR_region', ascending=False).reset_index(drop=True)
kpi1_region.index += 1
kpi1_region.columns = ['Región OMS','Confirmados','Muertes','CFR (%)']

print('KPI 1 — CASE FATALITY RATE (CFR)')
print('=' * 55)
print(f'  CFR Global     : {cfr_global:.3f}%')
print(f'  Total muertes  : {TOTAL_MUERTES:,}')
print(f'  Total confirmados: {TOTAL_CONFIRMADOS:,}')
print()
print('CFR por Región OMS:')
display(kpi1_region)

In [ ]:
region_order = kpi1_region['Región OMS'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Panel izquierdo: barras por región ───────────────────────────────────────
colores_cfr = ['crimson' if v > cfr_global else 'steelblue'
               for v in kpi1_region['CFR (%)']]
bars = axes[0].bar(kpi1_region['Región OMS'], kpi1_region['CFR (%)'],
                    color=colores_cfr, edgecolor='white', linewidth=0.8)
axes[0].axhline(cfr_global, color='navy', linestyle='--', linewidth=1.5,
                label=f'CFR Global: {cfr_global:.2f}%')
axes[0].axhline(2, color='orange', linestyle=':', linewidth=1.2, alpha=0.8,
                label='Umbral alerta (2%)')
axes[0].axhline(5, color='red', linestyle=':', linewidth=1.2, alpha=0.8,
                label='Umbral crítico (5%)')
for bar, val in zip(bars, kpi1_region['CFR (%)']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.2f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')
axes[0].set_title('KPI 1: CFR por Región OMS')
axes[0].set_ylabel('Case Fatality Rate (%)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=25)
axes[0].legend(fontsize=8)

# ── Panel derecho: top 15 países con mayor CFR (mín. 1,000 casos) ───────────
df_cfr_top = (
    df_pais[df_pais['Confirmed'] >= 1000]
    .nlargest(15, 'CFR')
    .sort_values('CFR')
)
colores_top = sns.color_palette('Reds_d', len(df_cfr_top))
axes[1].barh(df_cfr_top['Country/Region'], df_cfr_top['CFR'],
             color=colores_top, edgecolor='white')
axes[1].axvline(cfr_global, color='navy', linestyle='--', linewidth=1.5,
                label=f'CFR Global: {cfr_global:.2f}%')
for i, (val, reg) in enumerate(zip(df_cfr_top['CFR'], df_cfr_top['WHO Region'])):
    axes[1].text(val + 0.1, i, f'{val:.1f}%', va='center', fontsize=8)
axes[1].set_title('Top 15 Países con Mayor CFR\n(mín. 1,000 casos confirmados)')
axes[1].set_xlabel('CFR (%)')
axes[1].legend(fontsize=8)

plt.suptitle('KPI 1 — Tasa de Fatalidad de Casos (CFR)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../Outputs/kpi1_cfr.png', bbox_inches='tight', dpi=130)
plt.show()

region_max = kpi1_region.iloc[0]
region_min = kpi1_region.iloc[-1]
print(f'\n Interpretación:')
print(f'   El CFR global al cierre del dataset es {cfr_global:.2f}%, superando el umbral de alerta (2%).')
print(f'   La región con mayor CFR es {region_max["Región OMS"]} ({region_max["CFR (%)"]:.2f}%)')
print(f'   y la de menor CFR es {region_min["Región OMS"]} ({region_min["CFR (%)"]:.2f}%).')
print(f'   Esta diferencia de {region_max["CFR (%)"]-region_min["CFR (%)"]:.2f} pp fue estadísticamente')
print(f'   significativa según el ANOVA de la Fase 2 (F=2.32, p=0.046 < 0.05), confirmando H1.')

---
## KPI 2 — Tasa de Recuperación

**Definición:** Porcentaje de casos confirmados que evolucionaron favorablemente (alta médica o resolución).  
**Fórmula:**  
$$\text{Recovery Rate} = \frac{\text{Recovered}}{\text{Confirmed}} \times 100$$

**Justificación de negocio:**  
Complementa el CFR: un sistema con baja fatalidad y alta recuperación es eficiente. La tasa de recuperación refleja a la vez la calidad del sistema de salud y la exhaustividad del reporte epidemiológico. Países con alta recuperación son benchmarks de buenas prácticas clínicas (Alemania, Corea del Sur, China).

**Relación con hipótesis:**  
H1: si la CFR difiere entre regiones, se espera que la Recovery Rate también difiera de forma inversa. Regiones con alta CFR deberían mostrar menor Recovery Rate.

**Umbral de referencia:** Recovery Rate ≥ 70% = sistema efectivo; < 50% = alerta.

In [ ]:
# ── Cómputo del KPI 2 ────────────────────────────────────────────────────────
recovery_global = TOTAL_RECUPERADOS / TOTAL_CONFIRMADOS * 100

kpi2_region = df_region[['WHO Region','Confirmados','Recuperados','Recovery_region']].copy()
kpi2_region = kpi2_region.sort_values('Recovery_region', ascending=False).reset_index(drop=True)
kpi2_region.index += 1
kpi2_region.columns = ['Región OMS','Confirmados','Recuperados','Recovery Rate (%)']

print('KPI 2 — TASA DE RECUPERACIÓN')
print('=' * 55)
print(f'  Recovery Rate Global : {recovery_global:.2f}%')
print(f'  Total recuperados    : {TOTAL_RECUPERADOS:,}')
print()
print('Recovery Rate por Región OMS:')
display(kpi2_region)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Panel izquierdo: barras por región ───────────────────────────────────────
colores_rec = ['seagreen' if v >= 70 else ('orange' if v >= 50 else 'crimson')
               for v in kpi2_region['Recovery Rate (%)']]
r2_sorted = kpi2_region.sort_values('Recovery Rate (%)')
bars = axes[0].barh(r2_sorted['Región OMS'], r2_sorted['Recovery Rate (%)'],
                     color=colores_rec, edgecolor='white')
axes[0].axvline(recovery_global, color='navy', linestyle='--', linewidth=1.5,
                label=f'Global: {recovery_global:.1f}%')
axes[0].axvline(70, color='seagreen', linestyle=':', linewidth=1.2,
                label='Umbral efectivo (70%)')
axes[0].axvline(50, color='orange', linestyle=':', linewidth=1.2,
                label='Umbral alerta (50%)')
for bar, val in zip(bars, r2_sorted['Recovery Rate (%)']):
    axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)
axes[0].set_xlim(0, 110)
axes[0].set_title('KPI 2: Recovery Rate por Región OMS')
axes[0].set_xlabel('Recovery Rate (%)')
axes[0].legend(fontsize=8)

# ── Panel derecho: CFR vs Recovery Rate (relación inversa esperada) ──────────
palette_r = dict(zip(df_pais['WHO Region'].unique(),
                     sns.color_palette('muted', df_pais['WHO Region'].nunique())))
for region, grp in df_pais.groupby('WHO Region'):
    axes[1].scatter(grp['Recovery_Rate'], grp['CFR'],
                    color=palette_r[region], alpha=0.6, s=45, label=region)
rho, p_rho = stats.spearmanr(df_pais['Recovery_Rate'], df_pais['CFR'])
axes[1].set_title(f'CFR vs Recovery Rate por País\n(Spearman ρ = {rho:.3f}, p = {p_rho:.4f})')
axes[1].set_xlabel('Recovery Rate (%)')
axes[1].set_ylabel('CFR (%)')
axes[1].legend(fontsize=7, loc='upper right')

plt.suptitle('KPI 2 — Tasa de Recuperación',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../Outputs/kpi2_recovery.png', bbox_inches='tight', dpi=130)
plt.show()

r_max = kpi2_region.iloc[0]
r_min = kpi2_region.iloc[-1]
print(f'\n Interpretación:')
print(f'   La Recovery Rate global de {recovery_global:.1f}% supera el umbral de efectividad del 70%,')
print(f'   lo que indica que la mayoría de casos confirmados fueron resueltos favorablemente.')
print(f'   La correlación de Spearman entre Recovery Rate y CFR es ρ = {rho:.3f} (p = {p_rho:.4f}),')
print(f'   confirmando la relación inversa esperada: a mayor recuperación, menor fatalidad.')
print(f'   La región más eficiente es {r_max["Región OMS"]} ({r_max["Recovery Rate (%)"]:.1f}%) '
      f'y la de menor desempeño es {r_min["Región OMS"]} ({r_min["Recovery Rate (%)"]:.1f}%).')

---
## KPI 3 — Tasa de Casos Activos

**Definición:** Proporción de casos confirmados que se encuentran activos (sin resolver) al corte del dataset.  
**Fórmula:**  
$$\text{Active Rate} = \frac{\text{Active}}{\text{Confirmed}} \times 100$$

**Justificación de negocio:**  
Mide la **carga hospitalaria simultánea**: todos los casos activos son pacientes que demandan recursos al mismo tiempo (camas, oxígeno, personal médico). Una Active Rate elevada sostenida es señal de que el sistema no está logrando resolver los casos suficientemente rápido, lo que puede derivar en colapso sanitario. Es el indicador más relevante para la gestión de capacidad hospitalaria.

**Relación con hipótesis:**  
H1: si las regiones con mayor CFR también tienen mayor Active Rate, se refuerza la hipótesis de que la saturación del sistema es un factor explicativo de la mortalidad diferencial.

**Umbral de referencia:** Active Rate < 20% = situación manejable; 20–40% = presión alta; > 40% = riesgo de colapso.

In [ ]:
# ── Cómputo del KPI 3 ────────────────────────────────────────────────────────
active_global = df_pais['Active'].sum() / TOTAL_CONFIRMADOS * 100

print('KPI 3 — TASA DE CASOS ACTIVOS')
print('=' * 55)
print(f'  Active Rate Global : {active_global:.2f}%')
print(f'  Total activos      : {df_pais["Active"].sum():,}')
print()

kpi3_region = df_region[['WHO Region','Confirmados','Activos','Active_region']].copy()
kpi3_region = kpi3_region.sort_values('Active_region', ascending=False).reset_index(drop=True)
kpi3_region.index += 1
kpi3_region.columns = ['Región OMS','Confirmados','Activos','Active Rate (%)']
print('Active Rate por Región OMS:')
display(kpi3_region)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Panel izquierdo: composición apilada por región ──────────────────────────
r_comp = df_region.sort_values('Confirmados', ascending=False).copy()
r_comp['pct_rec']    = r_comp['Recuperados'] / r_comp['Confirmados'] * 100
r_comp['pct_active'] = r_comp['Activos']     / r_comp['Confirmados'] * 100
r_comp['pct_deaths'] = r_comp['Muertes']     / r_comp['Confirmados'] * 100

x = np.arange(len(r_comp))
w = 0.6
axes[0].bar(x, r_comp['pct_rec'],    w, label='Recuperados', color='seagreen', alpha=0.85)
axes[0].bar(x, r_comp['pct_active'], w, bottom=r_comp['pct_rec'],
            label='Activos', color='darkorange', alpha=0.85)
axes[0].bar(x, r_comp['pct_deaths'], w,
            bottom=r_comp['pct_rec'] + r_comp['pct_active'],
            label='Fallecidos', color='crimson', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(r_comp['WHO Region'], rotation=25, ha='right')
axes[0].set_title('KPI 3: Composición de Casos por Región\n(% sobre Confirmados)')
axes[0].set_ylabel('Porcentaje (%)')
axes[0].axhline(100, color='black', linewidth=0.5)
axes[0].legend(fontsize=8)

# ── Panel derecho: Active Rate vs CFR por país (H1) ──────────────────────────
for region, grp in df_pais.groupby('WHO Region'):
    axes[1].scatter(grp['Active_Rate'], grp['CFR'],
                    color=palette_r[region], alpha=0.6, s=45, label=region)
rho_ar, p_ar = stats.spearmanr(df_pais['Active_Rate'], df_pais['CFR'])
axes[1].axvline(20, color='orange', linestyle=':', linewidth=1.2, label='Umbral alerta (20%)')
axes[1].axvline(40, color='red',    linestyle=':', linewidth=1.2, label='Umbral colapso (40%)')
axes[1].set_title(f'Active Rate vs CFR por País\n(Spearman ρ = {rho_ar:.3f}, p = {p_ar:.4f})')
axes[1].set_xlabel('Active Rate (%)')
axes[1].set_ylabel('CFR (%)')
axes[1].legend(fontsize=7, loc='upper right')

plt.suptitle('KPI 3 — Tasa de Casos Activos (Carga Sanitaria)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../Outputs/kpi3_active.png', bbox_inches='tight', dpi=130)
plt.show()

n_criticos = (df_pais['Active_Rate'] > 40).sum()
print(f'\n Interpretación:')
print(f'   La Active Rate global de {active_global:.1f}% indica que al cierre del dataset')
print(f'   cerca de 1 de cada 3 casos diagnosticados seguía sin resolverse.')
print(f'   {n_criticos} países presentaban Active Rate > 40%, en zona de riesgo de colapso hospitalario.')
print(f'   La correlación con CFR (ρ = {rho_ar:.3f}, p = {p_ar:.4f}) refuerza que la saturación')
print(f'   del sistema es un factor asociado a la mayor mortalidad (H1).')

---
## KPI 4 — Índice de Mortalidad Relativa (IMR)

**Definición:** Número de muertes por cada 100 pacientes recuperados. Mide la proporción entre el volumen de casos fatales y el volumen de casos resueltos favorablemente.  
**Fórmula:**  
$$\text{IMR} = \frac{\text{Deaths}}{\text{Recovered}} \times 100$$

**Justificación de negocio:**  
A diferencia del CFR (que divide sobre el total de casos), el IMR compara directamente dos desenlaces excluyentes: morir vs. recuperarse. Un IMR de 5 significa que por cada 100 personas que se recuperan, 5 fallecen. Es especialmente útil para evaluar la eficiencia clínica real, independizándose del problema de subdiagnóstico (casos confirmados subreportados). Los países con mayor IMR son los que más necesitan reforzar su capacidad de cuidados intensivos.

**Relación con hipótesis:**  
H1 y H2: un IMR alto en regiones con llegada temprana del virus (como Europa) sugiere que la falta de preparación inicial elevó la mortalidad relativa.

**Umbral de referencia:** IMR < 3 = respuesta eficiente; 3–8 = moderado; > 8 = crítico.

In [ ]:
# ── Cómputo del KPI 4 ────────────────────────────────────────────────────────
imr_global = TOTAL_MUERTES / TOTAL_RECUPERADOS * 100

imr_region = df_region.copy()
imr_region['IMR_region'] = (imr_region['Muertes'] / imr_region['Recuperados'] * 100).round(4)

print('KPI 4 — ÍNDICE DE MORTALIDAD RELATIVA (IMR)')
print('=' * 55)
print(f'  IMR Global : {imr_global:.4f} (muertes por cada 100 recuperados)')
print()

kpi4_show = imr_region[['WHO Region','Muertes','Recuperados','IMR_region']].sort_values(
    'IMR_region', ascending=False).reset_index(drop=True)
kpi4_show.index += 1
kpi4_show.columns = ['Región OMS','Muertes','Recuperados','IMR']
display(kpi4_show)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Panel izquierdo: IMR por región ─────────────────────────────────────────
imr_sorted = imr_region.sort_values('IMR_region', ascending=False)
colores_imr = ['crimson' if v > 8 else ('orange' if v > 3 else 'seagreen')
               for v in imr_sorted['IMR_region']]
bars = axes[0].bar(imr_sorted['WHO Region'], imr_sorted['IMR_region'],
                    color=colores_imr, edgecolor='white')
axes[0].axhline(imr_global, color='navy', linestyle='--', linewidth=1.5,
                label=f'IMR Global: {imr_global:.2f}')
axes[0].axhline(8, color='red',    linestyle=':', linewidth=1.2, label='Umbral crítico (8)')
axes[0].axhline(3, color='orange', linestyle=':', linewidth=1.2, label='Umbral alerta (3)')
for bar, val in zip(bars, imr_sorted['IMR_region']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_title('KPI 4: IMR por Región OMS')
axes[0].set_ylabel('Muertes por cada 100 Recuperados')
axes[0].tick_params(axis='x', rotation=25)
axes[0].legend(fontsize=8)

# ── Panel derecho: IMR vs CFR por país ──────────────────────────────────────
df_imr_clean = df_pais.dropna(subset=['IMR'])
for region, grp in df_imr_clean.groupby('WHO Region'):
    axes[1].scatter(grp['IMR'], grp['CFR'],
                    color=palette_r[region], alpha=0.6, s=45, label=region)
# Línea de regresión
slope_i, intercept_i, r_i, p_i, _ = stats.linregress(df_imr_clean['IMR'], df_imr_clean['CFR'])
xv_i = np.linspace(0, df_imr_clean['IMR'].quantile(0.97), 200)
axes[1].plot(xv_i, intercept_i + slope_i * xv_i, 'k--', lw=1.5,
             label=f'Regresión (r={r_i:.3f})')
axes[1].set_xlim(left=0, right=df_imr_clean['IMR'].quantile(0.97))
axes[1].set_title('IMR vs CFR por País')
axes[1].set_xlabel('IMR (Muertes / 100 Recuperados)')
axes[1].set_ylabel('CFR (%)')
axes[1].legend(fontsize=7, loc='upper left')

plt.suptitle('KPI 4 — Índice de Mortalidad Relativa (IMR)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../Outputs/kpi4_imr.png', bbox_inches='tight', dpi=130)
plt.show()

print(f'\n Interpretación:')
print(f'   El IMR global de {imr_global:.2f} indica que por cada 100 pacientes recuperados,')
print(f'   {imr_global:.1f} fallecen. La correlación IMR-CFR (r = {r_i:.3f}) confirma que')
print(f'   ambas métricas capturan dimensiones complementarias de la mortalidad:')
print(f'   CFR mide el riesgo individual, IMR mide la eficiencia clínica del sistema.')

---
## KPI 5 — Velocidad de la Primera Ola (Días al Pico)

**Definición:** Número de días desde el primer caso registrado en cada país hasta el día en que se alcanzó el mayor número de nuevos confirmados diarios (pico de la primera ola).  
**Fórmula:**  
$$\text{Days to Peak} = \text{Fecha pico nuevos casos} - \text{Fecha primer caso}$$

**Justificación de negocio:**  
Captura la velocidad de propagación de la pandemia dentro de cada país. Países con un pico muy rápido (pocos días) tuvieron un crecimiento explosivo que dificultó la respuesta sanitaria. Países con pico tardío tuvieron más tiempo para preparar su sistema. Este KPI cuantifica la dimensión temporal de **H2** más allá de la simple correlación, y alimenta directamente la narrativa de las oleadas (Pregunta 2).

**Relación con hipótesis:**  
H2 (no confirmada por Pearson): aunque la correlación lineal con CFR no fue significativa, este KPI permite explorar si la velocidad de llegada al pico —no solo el momento de llegada— tiene relación con la mortalidad.

**Umbral de referencia:** < 30 días = propagación explosiva; 30–60 días = rápida; > 60 días = controlada.

In [ ]:
# ── Cómputo del KPI 5 ────────────────────────────────────────────────────────
# Media móvil 7 días de nuevos confirmados por país
df_ma7_pais = (
    df_hechos.groupby(['Country/Region','Date'])['New_Confirmed']
    .sum().reset_index()
    .sort_values(['Country/Region','Date'])
)
df_ma7_pais['MA7'] = (
    df_ma7_pais.groupby('Country/Region')['New_Confirmed']
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

# Fecha del pico (MA7 máxima) por país
peak_dates = (
    df_ma7_pais.groupby('Country/Region')
    .apply(lambda x: x.loc[x['MA7'].idxmax(), 'Date'])
    .reset_index()
    .rename(columns={0: 'Peak_Date'})
)

# Unir con primer caso y calcular días al pico
kpi5_df = first_case.merge(peak_dates, on='Country/Region')
kpi5_df['Days_to_Peak'] = (kpi5_df['Peak_Date'] - kpi5_df['First_Case_Date']).dt.days
kpi5_df = kpi5_df.merge(df_pais[['Country/Region','CFR','WHO Region']], on='Country/Region')
kpi5_df = kpi5_df[kpi5_df['Days_to_Peak'] >= 0]  # Excluye inconsistencias de reporte

print('KPI 5 — VELOCIDAD DE LA PRIMERA OLA (DÍAS AL PICO)')
print('=' * 55)
print(f'  Mediana global  : {kpi5_df["Days_to_Peak"].median():.0f} días')
print(f'  Media global    : {kpi5_df["Days_to_Peak"].mean():.1f} días')
print(f'  Mín / Máx       : {kpi5_df["Days_to_Peak"].min()} / {kpi5_df["Days_to_Peak"].max()} días')
print()
print('Países con pico más rápido (< 30 días):')
display(
    kpi5_df[kpi5_df['Days_to_Peak'] < 30]
    .sort_values('Days_to_Peak')[['Country/Region','WHO Region','Days_to_Peak','CFR']]
    .reset_index(drop=True).head(10)
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Panel izquierdo: distribución de días al pico por región ─────────────────
order_box = (
    kpi5_df.groupby('WHO Region')['Days_to_Peak']
    .median().sort_values().index.tolist()
)
sns.boxplot(data=kpi5_df, x='WHO Region', y='Days_to_Peak',
            order=order_box, palette='muted', ax=axes[0])
sns.stripplot(data=kpi5_df, x='WHO Region', y='Days_to_Peak',
              order=order_box, color='black', alpha=0.3, size=3,
              jitter=True, ax=axes[0])
axes[0].axhline(30, color='red',    linestyle='--', lw=1.2, label='Explosivo (<30d)')
axes[0].axhline(60, color='orange', linestyle='--', lw=1.2, label='Rápido (<60d)')
axes[0].axhline(kpi5_df['Days_to_Peak'].median(), color='navy',
                linestyle=':', lw=1.5,
                label=f'Mediana global ({kpi5_df["Days_to_Peak"].median():.0f}d)')
axes[0].set_title('KPI 5: Distribución de Días al Pico\npor Región OMS')
axes[0].set_ylabel('Días desde primer caso hasta pico')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=25)
axes[0].legend(fontsize=8)

# ── Panel derecho: Days_to_Peak vs CFR ──────────────────────────────────────
for region, grp in kpi5_df.groupby('WHO Region'):
    axes[1].scatter(grp['Days_to_Peak'], grp['CFR'],
                    color=palette_r.get(region, 'gray'), alpha=0.6, s=45, label=region)
slope_k5, intercept_k5, r_k5, p_k5, _ = stats.linregress(
    kpi5_df['Days_to_Peak'], kpi5_df['CFR']
)
xv_k5 = np.linspace(0, kpi5_df['Days_to_Peak'].max(), 200)
axes[1].plot(xv_k5, intercept_k5 + slope_k5 * xv_k5, 'k--', lw=1.5,
             label=f'Regresión (r = {r_k5:.3f}, p = {p_k5:.3f})')
axes[1].set_title('KPI 5: Días al Pico vs CFR por País')
axes[1].set_xlabel('Días desde primer caso hasta pico (MA7)')
axes[1].set_ylabel('CFR (%)')
axes[1].legend(fontsize=7)

plt.suptitle('KPI 5 — Velocidad de la Primera Ola (Días al Pico)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../Outputs/kpi5_days_to_peak.png', bbox_inches='tight', dpi=130)
plt.show()

n_explosivo = (kpi5_df['Days_to_Peak'] < 30).sum()
print(f'\n Interpretación:')
print(f'   La mediana global de {kpi5_df["Days_to_Peak"].median():.0f} días indica que la mayoría de países')
print(f'   tardó entre 1 y 2 meses en alcanzar el pico de su primera ola.')
print(f'   {n_explosivo} países tuvieron una propagación explosiva (< 30 días),')
print(f'   en su mayoría en Europa y las Américas.')
print(f'   La correlación con CFR (r = {r_k5:.3f}, p = {p_k5:.3f}) complementa el hallazgo')
print(f'   de H2: la velocidad de llegada al pico tiene una relación débil con la mortalidad.')

---
## KPI 6 — Media Móvil de Nuevos Casos (MA7)

**Definición:** Promedio de nuevos casos confirmados en los últimos 7 días, calculado a nivel global y por región. Permite suavizar la variabilidad del reporte diario (que presenta caídas los fines de semana) para identificar la tendencia real.

**Fórmula:**  
$$\text{MA7}_{t} = \frac{1}{7}\sum_{i=0}^{6} \text{New\_Confirmed}_{t-i}$$

**Justificación de negocio:**  
Es el indicador operativo más utilizado en los centros de gestión de crisis pandémica (OMS, CDC, ECDC) para monitorear la evolución en tiempo real. Permite identificar cuándo empieza una oleada, cuándo llega al pico y cuándo inicia la caída, lo que orienta decisiones de confinamiento, reapertura o despliegue de recursos. Responde directamente a la Pregunta 2 del proyecto.

**Relación con hipótesis:**  
H2: el patrón de oleadas secuenciales por región (Asia → Europa → Américas) es la evidencia visual más clara del efecto temporal que H2 intenta cuantificar.

**Umbral de referencia:** MA7 creciente = oleada en expansión (intervención requerida); MA7 decreciente = fase de control.

In [ ]:
# ── Cómputo del KPI 6 ────────────────────────────────────────────────────────
# MA7 global
global_daily = (
    df_hechos.groupby('Date')[['New_Confirmed','New_Deaths']]
    .sum().reset_index()
)
global_daily['MA7_conf']   = global_daily['New_Confirmed'].rolling(7, min_periods=1).mean()
global_daily['MA7_deaths'] = global_daily['New_Deaths'].rolling(7, min_periods=1).mean()

# MA7 por región
region_daily = (
    df_hechos.groupby(['WHO Region','Date'])['New_Confirmed']
    .sum().reset_index()
    .sort_values(['WHO Region','Date'])
)
region_daily['MA7'] = (
    region_daily.groupby('WHO Region')['New_Confirmed']
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

# Valor en el último día disponible
ma7_ultimo = global_daily.iloc[-1]
print('KPI 6 — MEDIA MÓVIL DE NUEVOS CASOS (MA7)')
print('=' * 55)
print(f'  MA7 global al cierre ({latest_date.date()}):')
print(f'    Nuevos confirmados : {ma7_ultimo["MA7_conf"]:,.0f} por día')
print(f'    Nuevas muertes     : {ma7_ultimo["MA7_deaths"]:,.0f} por día')
print()
print('MA7 al cierre por región:')
ma7_region_cierre = (
    region_daily[region_daily['Date'] == latest_date]
    [['WHO Region','MA7']].sort_values('MA7', ascending=False)
    .reset_index(drop=True)
)
ma7_region_cierre.index += 1
ma7_region_cierre.columns = ['Región OMS','MA7 nuevos casos/día']
display(ma7_region_cierre)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# ── Panel superior: MA7 global + barras diarias ───────────────────────────────
axes[0].bar(global_daily['Date'], global_daily['New_Confirmed'] / 1e3,
            color='steelblue', alpha=0.35, width=1, label='Nuevos casos diarios')
axes[0].plot(global_daily['Date'], global_daily['MA7_conf'] / 1e3,
             color='navy', linewidth=2, label='MA7 Nuevos casos')
ax_d = axes[0].twinx()
ax_d.plot(global_daily['Date'], global_daily['MA7_deaths'],
          color='crimson', linewidth=1.8, linestyle='--', label='MA7 Nuevas muertes')
ax_d.set_ylabel('MA7 Nuevas muertes / día', color='crimson')
ax_d.tick_params(axis='y', labelcolor='crimson')
axes[0].set_title('KPI 6: Evolución Global — MA7 de Nuevos Confirmados y Muertes')
axes[0].set_ylabel('Miles de nuevos casos / día')
lines1, lbl1 = axes[0].get_legend_handles_labels()
lines2, lbl2 = ax_d.get_legend_handles_labels()
axes[0].legend(lines1 + lines2, lbl1 + lbl2, fontsize=8, loc='upper left')
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(8))
axes[0].tick_params(axis='x', rotation=20)

# ── Panel inferior: MA7 por región (oleadas secuenciales) ────────────────────
region_palette = dict(zip(
    region_daily['WHO Region'].unique(),
    sns.color_palette('tab10', region_daily['WHO Region'].nunique())
))
for region, grp in region_daily.groupby('WHO Region'):
    axes[1].plot(grp['Date'], grp['MA7'] / 1e3,
                 color=region_palette[region], linewidth=1.8, label=region)
axes[1].set_title('KPI 6: MA7 de Nuevos Casos por Región OMS — Oleadas Secuenciales')
axes[1].set_ylabel('Miles de nuevos casos / día (MA7)')
axes[1].legend(fontsize=8, loc='upper left')
axes[1].xaxis.set_major_locator(mticker.MaxNLocator(8))
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('KPI 6 — Media Móvil de Nuevos Casos (MA7)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../Outputs/kpi6_ma7.png', bbox_inches='tight', dpi=130)
plt.show()

pico_global = global_daily.loc[global_daily['MA7_conf'].idxmax()]
print(f'\n Interpretación:')
print(f'   El pico global de la MA7 se alcanzó el {pico_global["Date"].date()}')
print(f'   con {pico_global["MA7_conf"]:,.0f} nuevos casos promedio por día.')
print(f'   El gráfico por región evidencia el patrón de oleadas secuenciales identificado')
print(f'   en la Fase 1: Asia lideró la primera fase, seguida por Europa y las Américas,')
print(f'   confirmando visualmente la hipótesis H2 aunque sin significancia estadística lineal.')

---
## KPI 7 — Concentración de la Carga Global (Curva de Lorenz + Índice de Gini)

**Definición:** Mide cuán desigualmente distribuida está la carga pandémica entre los países del mundo. Un Gini = 0 implica que todos los países tienen exactamente el mismo número de casos; Gini = 1 implica que un único país concentra todos los casos.

**Fórmula:**  
$$G = \frac{2\sum_{i=1}^{n} i \cdot x_i}{n \sum_{i=1}^{n} x_i} - \frac{n+1}{n}, \quad x_i \text{ ordenado de menor a mayor}$$

**Justificación de negocio:**  
Si la pandemia está altamente concentrada en pocos países, las estrategias globales de respuesta (distribución de vacunas, cooperación técnica, financiamiento de emergencia) deben priorizarse hacia esos países. Este indicador orienta la **asignación eficiente de recursos internacionales** y permite identificar si los países con mayor carga son también los de mayor mortalidad (lo que reforzaría H1).

**Relación con hipótesis:**  
H1 y H2 transversalmente: si la concentración de casos coincide con regiones de alta CFR, se justifica una respuesta diferenciada. Si la concentración es alta, los promedios globales están dominados por los outliers identificados en la Fase 3.

**Interpretación del Gini:** < 0.5 = distribución moderada; 0.5–0.8 = alta concentración; > 0.8 = concentración extrema.

In [ ]:
# ── Función Gini ─────────────────────────────────────────────────────────────
def gini_coefficient(array):
    """Calcula el coeficiente de Gini para un arreglo 1-D de valores no negativos."""
    x = np.sort(np.abs(array.astype(float)))
    n = len(x)
    index = np.arange(1, n + 1)
    return float((2 * np.dot(index, x)) / (n * x.sum()) - (n + 1) / n)

# ── Cómputo del KPI 7 ────────────────────────────────────────────────────────
gini_confirmed = gini_coefficient(df_pais['Confirmed'].values)
gini_deaths    = gini_coefficient(df_pais['Deaths'].values)
gini_recovered = gini_coefficient(df_pais['Recovered'].values)

# ¿Cuántos países concentran el 80% de los casos?
df_sorted = df_pais.sort_values('Confirmed', ascending=False).reset_index(drop=True)
df_sorted['cum_pct'] = df_sorted['Confirmed'].cumsum() / TOTAL_CONFIRMADOS * 100
n_80 = int((df_sorted['cum_pct'] <= 80).sum()) + 1
pct_paises_80 = n_80 / len(df_pais) * 100
top10_pct = df_sorted.head(10)['Confirmed'].sum() / TOTAL_CONFIRMADOS * 100

print('KPI 7 — CONCENTRACIÓN DE LA CARGA GLOBAL (GINI)')
print('=' * 55)
print(f'  Gini — Confirmados : {gini_confirmed:.4f}')
print(f'  Gini — Muertes     : {gini_deaths:.4f}')
print(f'  Gini — Recuperados : {gini_recovered:.4f}')
print()
print(f'  El {pct_paises_80:.1f}% de los países ({n_80} de {len(df_pais)}) concentra el 80% de los casos.')
print(f'  Los 10 países más afectados representan el {top10_pct:.1f}% del total global.')
print()
print('Top 10 países con mayor carga:')
display(
    df_sorted.head(10)[['Country/Region','WHO Region','Confirmed','Deaths','CFR','cum_pct']]
    .rename(columns={'cum_pct':'% Acumulado global'})
    .reset_index(drop=True)
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Panel izquierdo: Curva de Lorenz ─────────────────────────────────────────
vals_conf = np.sort(df_pais['Confirmed'].values)
vals_death = np.sort(df_pais['Deaths'].values)
cum_conf  = np.cumsum(vals_conf)  / vals_conf.sum()
cum_death = np.cumsum(vals_death) / vals_death.sum()
x_lorenz  = np.linspace(0, 1, len(cum_conf))

axes[0].plot([0,1],[0,1], '--', color='gray', lw=1.2, label='Igualdad perfecta')
axes[0].plot(x_lorenz, cum_conf,  color='steelblue', lw=2,
             label=f'Confirmados (Gini={gini_confirmed:.3f})')
axes[0].plot(x_lorenz, cum_death, color='crimson',   lw=2, linestyle='--',
             label=f'Muertes (Gini={gini_deaths:.3f})')
axes[0].fill_between(x_lorenz, cum_conf, x_lorenz, alpha=0.15, color='steelblue')
axes[0].fill_between(x_lorenz, cum_death, x_lorenz, alpha=0.15, color='crimson')
axes[0].set_title('KPI 7: Curva de Lorenz\nConcentración de Casos y Muertes')
axes[0].set_xlabel('Proporción acumulada de países')
axes[0].set_ylabel('Proporción acumulada de casos / muertes')
axes[0].legend(fontsize=9)

# ── Panel derecho: Diagrama de Pareto (Top 20 países) ────────────────────────
df_pareto = df_sorted.head(20).copy()
bar_colors = [region_palette.get(r, 'gray') for r in df_pareto['WHO Region']]
axes[1].bar(range(len(df_pareto)), df_pareto['Confirmed'] / 1e6,
            color=bar_colors, alpha=0.85, label='Confirmados (M)')
ax_pareto = axes[1].twinx()
ax_pareto.plot(range(len(df_pareto)), df_pareto['cum_pct'],
               color='black', marker='o', markersize=4, lw=2, label='% Acumulado')
ax_pareto.axhline(80, color='red', linestyle='--', lw=1.2, alpha=0.7)
ax_pareto.set_ylabel('% Acumulado del total global', color='black')
ax_pareto.set_ylim(0, 105)
axes[1].set_xticks(range(len(df_pareto)))
axes[1].set_xticklabels(df_pareto['Country/Region'], rotation=45, ha='right', fontsize=7.5)
axes[1].set_title('KPI 7: Concentración — Top 20 Países (Pareto)')
axes[1].set_ylabel('Casos Confirmados (millones)')
lines_b, lbl_b = axes[1].get_legend_handles_labels()
lines_p, lbl_p = ax_pareto.get_legend_handles_labels()
axes[1].legend(lines_b + lines_p, lbl_b + lbl_p, fontsize=8, loc='upper left')

plt.suptitle('KPI 7 — Concentración de la Carga Global (Índice de Gini)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../Outputs/kpi7_gini.png', bbox_inches='tight', dpi=130)
plt.show()

nivel_conc = 'alta' if gini_confirmed > 0.5 else 'moderada'
print(f'\n Interpretación:')
print(f'   Un Gini de {gini_confirmed:.3f} sobre los casos confirmados indica una concentración {nivel_conc}.')
print(f'   Solo {n_80} países ({pct_paises_80:.0f}% del total) concentran el 80% de los casos globales.')
print(f'   El Gini de muertes ({gini_deaths:.3f}) es {'similar' if abs(gini_deaths-gini_confirmed)<0.05 else 'mayor'},')
print(f'   lo que indica que la mortalidad está {'igualmente' if abs(gini_deaths-gini_confirmed)<0.05 else 'más'} concentrada que los casos.')
print(f'   Esto refuerza la relevancia de H1: la carga pandémica no es uniforme y los análisis')
print(f'   regionales están fuertemente influenciados por los países de mayor volumen.')

---
## 3. Resumen Ejecutivo de KPIs

In [ ]:
# ── Tabla resumen de todos los KPIs ──────────────────────────────────────────
resumen_kpis = pd.DataFrame([
    {
        'KPI': 'KPI 1 — CFR Global',
        'Fórmula': 'Deaths / Confirmed × 100',
        'Valor Global': f'{cfr_global:.3f}%',
        'Umbral / Referencia': '< 2% efectivo | > 5% crítico',
        'Hipótesis': 'H1, H2',
        'Estado': ' Alerta' if cfr_global > 2 else ' OK',
    },
    {
        'KPI': 'KPI 2 — Recovery Rate',
        'Fórmula': 'Recovered / Confirmed × 100',
        'Valor Global': f'{recovery_global:.2f}%',
        'Umbral / Referencia': '≥ 70% efectivo | < 50% alerta',
        'Hipótesis': 'H1',
        'Estado': ' Efectivo' if recovery_global >= 70 else ' En riesgo',
    },
    {
        'KPI': 'KPI 3 — Active Rate',
        'Fórmula': 'Active / Confirmed × 100',
        'Valor Global': f'{active_global:.2f}%',
        'Umbral / Referencia': '< 20% manejable | > 40% colapso',
        'Hipótesis': 'H1',
        'Estado': ' Presión alta' if active_global > 20 else ' OK',
    },
    {
        'KPI': 'KPI 4 — IMR',
        'Fórmula': 'Deaths / Recovered × 100',
        'Valor Global': f'{imr_global:.4f}',
        'Umbral / Referencia': '< 3 eficiente | > 8 crítico',
        'Hipótesis': 'H1, H2',
        'Estado': ' Crítico' if imr_global > 8 else (' Moderado' if imr_global > 3 else ' OK'),
    },
    {
        'KPI': 'KPI 5 — Días al Pico',
        'Fórmula': 'Peak_Date − First_Case_Date',
        'Valor Global': f'{kpi5_df["Days_to_Peak"].median():.0f} días (mediana)',
        'Umbral / Referencia': '< 30d explosivo | > 60d controlado',
        'Hipótesis': 'H2',
        'Estado': ' Rápido',
    },
    {
        'KPI': 'KPI 6 — MA7 Nuevos Casos',
        'Fórmula': 'Promedio 7 días de New_Confirmed',
        'Valor Global': f'{pico_global["MA7_conf"]:,.0f}/día (pico)',
        'Umbral / Referencia': 'Tendencia creciente = oleada activa',
        'Hipótesis': 'H2',
        'Estado': ' Monitoreo',
    },
    {
        'KPI': 'KPI 7 — Índice de Gini',
        'Fórmula': 'Coeficiente de Gini sobre Confirmed',
        'Valor Global': f'{gini_confirmed:.4f}',
        'Umbral / Referencia': '< 0.5 moderado | > 0.8 extremo',
        'Hipótesis': 'H1, H2',
        'Estado': ' Alta concentr.' if gini_confirmed > 0.5 else ' Moderada',
    },
])

print('RESUMEN EJECUTIVO DE KPIs — COVID-19 DATASET')
print('=' * 110)
display(resumen_kpis)

import os
os.makedirs('../Outputs', exist_ok=True)
resumen_kpis.to_csv('../Outputs/resumen_kpis.csv', index=False)
print('\n Tabla de resumen exportada: Outputs/resumen_kpis.csv')

---
## 4. Conclusiones de la Fase 4

### 4.1 Hallazgos por KPI

| KPI | Hallazgo principal | Implicación para decisiones |
|---|---|---|
| **KPI 1 — CFR** | El CFR global supera el umbral de alerta (2%), con diferencias significativas entre regiones (H1 confirmada). Europa y las Américas lideran la mortalidad relativa. | Priorizar refuerzo de diagnóstico y UCI en regiones de alta CFR. |
| **KPI 2 — Recovery Rate** | La Recovery Rate global supera el 70% al corte, con correlación negativa significativa con CFR (a mayor recuperación, menor fatalidad). | Replicar protocolos de países con alta Recovery Rate (Alemania, Corea del Sur) como benchmarks. |
| **KPI 3 — Active Rate** | La Active Rate global supera el 20%, indicando alta carga simultánea al cierre. Varios países entran en zona de riesgo de colapso (>40%). | Activar protocolos de ampliación de capacidad hospitalaria preventiva. |
| **KPI 4 — IMR** | El IMR confirma que la mortalidad relativa varía entre regiones, independientemente del volumen de casos. El IMR es más sensible que el CFR al subdiagnóstico. | Usar el IMR como indicador de alerta temprana cuando los sistemas de reporte son incompletos. |
| **KPI 5 — Días al Pico** | La mediana global de días al pico muestra que los países de Europa y las Américas alcanzaron el pico más rápido, coherente con H2. La correlación con CFR es débil pero en la dirección esperada. | Implementar medidas de contención en las primeras 2–4 semanas del brote para ralentizar la llegada al pico. |
| **KPI 6 — MA7** | El gráfico por región confirma visualmente el patrón de oleadas secuenciales: Asia → Europa → Américas. La MA7 es el indicador operativo clave para decisiones de confinamiento y reapertura. | Monitorear la MA7 semanal por región como semáforo epidemiológico en el dashboard. |
| **KPI 7 — Gini** | La distribución de casos es altamente desigual: pocos países concentran la mayoría de la carga. El Gini de muertes es similar al de casos. | Las estrategias globales deben focalizarse en los 10–15 países de mayor carga para maximizar impacto. |

### 4.2 Vínculo entre KPIs e hipótesis

**H1 — La CFR difiere entre regiones (CONFIRMADA):**  
Los KPIs 1, 2, 3 y 4 operacionalizan esta hipótesis. La diferencia entre regiones es consistente en todos los indicadores: Europa y las Américas presentan mayor CFR, menor Recovery Rate y mayor IMR que Asia-Pacífico y África. Esto sugiere que los factores explicativos van más allá del timing de llegada e incluyen capacidad diagnóstica, acceso a UCI y protocolos terapéuticos.

**H2 — El timing de llegada correlaciona con CFR (NO CONFIRMADA, p = 0.055):**  
Los KPIs 5 y 6 enriquecen el análisis de H2. Aunque la correlación lineal no fue significativa, el KPI 5 muestra que los países con picos más rápidos tienden a ser los de mayor CFR, y el KPI 6 evidencia visualmente el patrón de oleadas secuenciales. El efecto existe en la dirección esperada, pero no alcanza significancia estadística con esta muestra y esta especificación lineal.